# Swarm-MeZO — Day 5: trimmed-mean consensus on RoBERTa+SST-2 (Colab)

Запускает `scripts/run_trimmed.py` на GPU в Colab — контрольная ветка robust-aggregation.

**Идея.** Day 4: репутация мягко *понижает вес* плохих агентов и каскадит. Trimmed-mean — устойчивая форма «уйти от плохих агентов»: худшие `trim_k` агентов по probe-лоссу *жёстко выбрасываются* из центроида, `W` остаётся row-stochastic (в отличие от отталкивания с `W_ij<0`, которое расходится). Семейство Byzantine-robust агрегаторов — trimmed mean / Krum, Yin et al. 2018. Модификатор `trim_k` композится с β: `β=0` → чистый trimmed mean (равномерно по выжившим), `β>0` → репутация среди выживших.

**Прогон (только IID** — non-IID по теории вне области применимости): N=8 агентов, IID-split, K=100 локальных MeZO-шагов между раундами, 5000 шагов; сетка `β ∈ {0, 0.1, 0.5, 1, 10} × trim_k ∈ {2, 4}` = **10 прогонов**.

**Бейзлайн** (trim_k=0) — это существующий IID-прогон репутации `outputs/day5_reputation_iid.json`.

**Время:** ≈1.5ч на конфиг (L4/A100, bf16), ≈3ч на T4. Итого 10 конфигов = 15–30 часов — рассчитывайте на несколько сессий (идемпотентно).

**Перед запуском:** (1) запушить ветку с `run_trimmed.py` и правками `src/` в GitHub; (2) Runtime → Change runtime type → GPU.

## 1. Setup: клонирование и зависимости

In [ ]:
!nvidia-smi | head -20

In [ ]:
# Клонируем репозиторий (публичный). git pull подтягивает свежий run_trimmed.py.
import os
REPO_DIR = '/content/swarm-mezo'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/olegkeatzin/swarm-mezo.git $REPO_DIR
else:
    !cd $REPO_DIR && git pull
%cd $REPO_DIR

In [ ]:
!pip install -q 'transformers>=4.40' 'datasets>=2.20' tqdm matplotlib pandas
import torch
print(f'torch {torch.__version__}, cuda available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'gpu: {torch.cuda.get_device_name(0)}')
    cap = torch.cuda.get_device_capability(0)
    print(f'compute capability: {cap}  (bf16 fast on cap >= (8,0))')

## 2. Прогрев кеша: модель + датасет

Скачивает RoBERTa-base и SST-2 в `~/.cache/huggingface`.

In [ ]:
from datasets import load_dataset            # ВАЖНО: до torch
from transformers import AutoModelForMaskedLM, AutoTokenizer

_ = load_dataset('glue', 'sst2')
_ = AutoTokenizer.from_pretrained('roberta-base')
_ = AutoModelForMaskedLM.from_pretrained('roberta-base')
print('cache OK')

## 3. Санитарный smoke-test (≈1–2 минуты)

Юнит-тесты trim_k-математики (`tests/test_reputation.py`) + vmap/bf16-путь консенсуса на GPU (`smoke_test_reputation.py` — тот же forward, что у trimmed-прогона).

In [ ]:
!python -m pytest tests/test_reputation.py -q 2>&1 | tail -8
!python scripts/smoke_test_reputation.py 2>&1 | tail -20

## 4. Главный прогон

Идемпотентный — пишет в `outputs/day5_trimmed.json` после каждого `(β, trim_k)`. Разрыв сессии → перезапуск этой ячейки продолжит с недосчитанного.

In [ ]:
!mkdir -p outputs && python -u scripts/run_trimmed.py 2>&1 | tee -a outputs/day5_trimmed.log

## 5. Быстрый превью результатов

Финальная val accuracy по сетке β × trim_k. Полный анализ — в отдельном notebook позже.

In [ ]:
import json
import matplotlib.pyplot as plt

d = json.loads(open('outputs/day5_trimmed.json').read())
runs = d['runs']
betas = sorted({h['beta'] for h in runs.values()})
trims = sorted({h['trim_k'] for h in runs.values()})
print(f'{len(runs)} configs  |  β={betas}  trim_k={trims}')
print()
hdr = 'β \\ trim_k'
print(f'{hdr:>12} | ' + ' | '.join(f'k={k:>6}' for k in trims))
for b in betas:
    cells = []
    for k in trims:
        h = runs.get(f'beta{b}_trim{k}')
        cells.append(f'{h["eval_acc"][-1]:.4f}' if h else '   —  ')
    print(f'{b:>12} | ' + ' | '.join(f'{c:>8}' for c in cells))

In [ ]:
# Кривые val accuracy — одна линия на (β, trim_k)
fig, axes = plt.subplots(1, len(trims), figsize=(6 * len(trims), 5),
                         dpi=110, sharey=True, squeeze=False)
cmap = plt.get_cmap('viridis')
for ax, k in zip(axes[0], trims):
    sub = {b: runs.get(f'beta{b}_trim{k}') for b in betas}
    sub = {b: h for b, h in sub.items() if h}
    for i, (b, h) in enumerate(sub.items()):
        color = cmap(i / max(len(sub) - 1, 1))
        ax.plot(h['eval_step'], h['eval_acc'], marker='o', markersize=3,
                color=color, label=f'β = {b}')
    ax.set_title(f'trim_k = {k}  (drop worst {k}/8)')
    ax.set_xlabel('per-agent step'); ax.grid(True, alpha=0.3); ax.legend()
axes[0][0].set_ylabel('SST-2 val accuracy')
fig.suptitle('Day 5: trimmed-mean MeZO on RoBERTa+SST-2 (IID)')
plt.tight_layout(); plt.show()

## 6. Скачивание результатов

In [ ]:
from google.colab import files
files.download('outputs/day5_trimmed.json')
files.download('outputs/day5_trimmed.log')